[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-3-nlp-to-transformers/04-pretrained-transformers/code/using_pretrained_transformers.ipynb)

# Class 3.4: Using pretrained transformers

You understand the transformer block. Here you stop building and start using: load a real pretrained model from Hugging Face and run inference. Each cell shows the expected output as a comment, so you can check your run.

What we will cover:
- loading a tokenizer and model together
- the one-call `pipeline` for common tasks
- reading a classification output by hand (logits, softmax, id2label)
- sentence embeddings from a model
- batched inference in eval mode


## Setup

You need Hugging Face `transformers` for this notebook (PyTorch is already installed from class 2.4, PyTorch fundamentals). The first run downloads the model, so an internet connection is required. To install it, run:

```
pip install transformers
```


## Load a tokenizer and model together

Always load both with the same name. The tokenizer must be the one the model was trained with.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

name = "distilbert-base-uncased"      # a small encoder, downloads fast
tok = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name)

inputs = tok("this is our first example", return_tensors="pt")
print("input ids:", inputs["input_ids"])           # -> tensor([[ 101, 2023, 2003, 2256, 2034, 2742, 102]])
print("tokens:", tok.convert_ids_to_tokens(inputs["input_ids"][0]))
# -> ['[CLS]', 'this', 'is', 'our', 'first', 'example', '[SEP]']

model.eval()
with torch.no_grad():
    out = model(**inputs)
print("hidden state shape:", tuple(out.last_hidden_state.shape))   # -> (1, 7, 768)

c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sourav Karmakar\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either ne

input ids: tensor([[ 101, 2023, 2003, 2256, 2034, 2742,  102]])
tokens: ['[CLS]', 'this', 'is', 'our', 'first', 'example', '[SEP]']
hidden state shape: (1, 7, 768)


In [4]:
inputs = tok(["this is our first example. I am using huggingface", "this is another example where I am using BERT"], padding=True, return_tensors="pt")

inputs

{'input_ids': tensor([[  101,  2023,  2003,  2256,  2034,  2742,  1012,  1045,  2572,  2478,
         17662, 12172,   102],
        [  101,  2023,  2003,  2178,  2742,  2073,  1045,  2572,  2478, 14324,
           102,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])}

In [6]:
for i in range(len(inputs["input_ids"])):
    print(f"Example {i}:")
    print("input ids:", inputs["input_ids"][i])
    print("tokens:", tok.convert_ids_to_tokens(inputs["input_ids"][i]))
    print()

Example 0:
input ids: tensor([  101,  2023,  2003,  2256,  2034,  2742,  1012,  1045,  2572,  2478,
        17662, 12172,   102])
tokens: ['[CLS]', 'this', 'is', 'our', 'first', 'example', '.', 'i', 'am', 'using', 'hugging', '##face', '[SEP]']

Example 1:
input ids: tensor([  101,  2023,  2003,  2178,  2742,  2073,  1045,  2572,  2478, 14324,
          102,     0,     0])
tokens: ['[CLS]', 'this', 'is', 'another', 'example', 'where', 'i', 'am', 'using', 'bert', '[SEP]', '[PAD]', '[PAD]']



In [7]:
tok.special_tokens_map

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

## A pipeline: one call for a common task

The `pipeline` helper wraps tokenize, run, and decode. Great for a quick result.

In [ ]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")
print(clf("I loved this class"))
# -> [{'label': 'POSITIVE', 'score': 0.9998...}]
print(clf("this was confusing and slow"))
# -> [{'label': 'NEGATIVE', 'score': 0.99...}]

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sourav Karmakar\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to 

[{'label': 'POSITIVE', 'score': 0.9998722076416016}]
[{'label': 'NEGATIVE', 'score': 0.9996939897537231}]


## Under the pipeline: logits to a label by hand

A pipeline just wraps a few lines. Load a classification model, read its raw logits, softmax them, and name the winner with `id2label`, the same steps as the slide.

In [9]:
from transformers import AutoModelForSequenceClassification

sent_name = "distilbert-base-uncased-finetuned-sst-2-english"
sent_tok = AutoTokenizer.from_pretrained(sent_name)
sent_model = AutoModelForSequenceClassification.from_pretrained(sent_name)

enc = sent_tok("I loved this class", return_tensors="pt")

sent_model.eval()
with torch.no_grad():
    logits = sent_model(**enc).logits[0]
probs = logits.softmax(-1)
label_id = int(probs.argmax())
print("label:", sent_model.config.id2label[label_id])    # -> POSITIVE
print("confidence:", round(float(probs[label_id]), 4))    # -> ~0.9999

c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sourav Karmakar\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 1

label: POSITIVE
confidence: 0.9999


## Sentence embeddings

The model's hidden states are vectors that capture meaning. A common trick is to average the token vectors into one sentence vector (the representation idea from class 2.5, Training in practice).

In [11]:
def embed(text):
    enc = tok(text, return_tensors="pt")
    with torch.no_grad():
        h = model(**enc).last_hidden_state          # (1, tokens, 768)
    return h.mean(dim=1).squeeze()                  # average -> (768,)

v1 = embed("The cat sat on the mat")
v2 = embed("The dog sat on the log")

cos = torch.nn.functional.cosine_similarity(v1, v2, dim=0)
print("embedding size:", tuple(v1.shape))           # -> (768,)
print("cosine similarity of two similar sentences:", round(float(cos), 3))
# -> a high value (close sentences sit close in vector space)

embedding size: (768,)
cosine similarity of two similar sentences: 0.932


## Batched inference

Pass several texts at once as a padded batch. Faster than one at a time, and the batching idea is the same as class 2.5 (Training in practice).

In [12]:
texts = ["first sentence", "a slightly longer second one", "third"]
batch = tok(texts, padding=True, return_tensors="pt")
print("batch input shape:", tuple(batch["input_ids"].shape))   # -> (3, max_len)
with torch.no_grad():
    out = model(**batch)
print("batched hidden shape:", tuple(out.last_hidden_state.shape))   # -> (3, max_len, 768)

batch input shape: (3, 7)
batched hidden shape: (3, 7, 768)


## Your turn

**Micro-assignment.** Six problems on using pretrained models; see `../micro-assignment/README.md`.

**Next, class 3.5 (From models to LLMs):** how these models scale into LLMs, and how generation is controlled with temperature and top-p.